<a href="https://colab.research.google.com/github/aisyahkurnia-25/UTS-Prediksi-Modern-dan-Machine-Learning/blob/main/UTS_Prediksi_Modern_%26_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **UTS PREDIKSI MODERN & MACHINE LEARNING**

Experiment Challenge: Credit Default Prediction

## **SETUP & DOWNLOAD DATASET**

In [1]:
# Import Library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Unggah file UCI_Credit_Card.csv ke Colab dan baca dataset
df = pd.read_csv('UCI_Credit_Card.csv')

# List untuk menampung ringkasan hasil eksperimen
summary_results = []

## **EXPERIMENT 0: DATA UNDERSTANDING & PREPARATION**

In [2]:
print("--- EXPERIMENT 0: DATA UNDERSTANDING & PREPARATION ---")
print(f"Dimensi Dataset: {df.shape}")
print("\nMissing Values:\n", df.isnull().sum().sum())
print("\nDuplikat Data:\n", df.duplicated().sum())

# Cek distribusi target
target_col = 'default.payment.next.month'
print("\nDistribusi Target:")
print(df[target_col].value_counts(normalize=True))

# Kebersihan data: kategori abnormal pada EDUCATION & MARRIAGE
df['EDUCATION'] = df['EDUCATION'].replace([0, 5, 6], 4)
df['MARRIAGE'] = df['MARRIAGE'].replace(0, 3)

# Pemisahan Predictor (X) dan Target (y) - Mengeluarkan 'ID'
X = df.drop(columns=['ID', target_col])
y = df[target_col]

# Train-Test Split (80:20) dengan Stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nUkuran Data Latih: {X_train.shape}")
print(f"Ukuran Data Uji: {X_test.shape}")

# Setup Cross Validation K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X_tr, y_tr, X_te, y_te, exp_name, model_name):
    # Cross Validation pada Training Set
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='f1')
    cv_f1 = np.mean(cv_scores)

    # Fit dan Prediksi pada Test Set
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    acc = accuracy_score(y_te, y_pred)
    prec = precision_score(y_te, y_pred)
    rec = recall_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred)

    summary_results.append({
        'Experiment': exp_name,
        'Model': model_name,
        'CV F1': round(cv_f1, 4),
        'Test Accuracy': round(acc, 4),
        'Test Precision': round(prec, 4),
        'Test Recall': round(rec, 4),
        'Test F1': round(f1, 4)
    })

    print(f"[{exp_name} - {model_name}] CV F1: {cv_f1:.4f} | Test F1: {f1:.4f}")

--- EXPERIMENT 0: DATA UNDERSTANDING & PREPARATION ---
Dimensi Dataset: (30000, 25)

Missing Values:
 0

Duplikat Data:
 0

Distribusi Target:
default.payment.next.month
0    0.7788
1    0.2212
Name: proportion, dtype: float64

Ukuran Data Latih: (24000, 23)
Ukuran Data Uji: (6000, 23)


## **EXPERIMENT 1: BASELINE MODEL**

In [3]:
print("\n--- EXPERIMENT 1: BASELINE MODEL ---")
logreg_baseline = LogisticRegression(max_iter=1000, random_state=42)
evaluate_model(logreg_baseline, X_train, y_train, X_test, y_test, "1", "Baseline (Logistic Regression)")


--- EXPERIMENT 1: BASELINE MODEL ---


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

[1 - Baseline (Logistic Regression)] CV F1: 0.3367 | Test F1: 0.3815


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## **EXPERIMENT 2: PREPROCESSING**

In [4]:
print("\n--- EXPERIMENT 2: PREPROCESSING ---")
# Menggunakan Pipeline untuk mencegah data leakage
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
evaluate_model(logreg_pipeline, X_train, y_train, X_test, y_test, "2", "Baseline + Preprocessing")


--- EXPERIMENT 2: PREPROCESSING ---
[2 - Baseline + Preprocessing] CV F1: 0.3662 | Test F1: 0.3583


## **EXPERIMENT 3: MODEL COMPARISON**

In [5]:
print("\n--- EXPERIMENT 3: MODEL COMPARISON ---")
models = {
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000, random_state=42))]),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}

for name, model in models.items():
    evaluate_model(model, X_train, y_train, X_test, y_test, "3", name)


--- EXPERIMENT 3: MODEL COMPARISON ---
[3 - Logistic Regression] CV F1: 0.3662 | Test F1: 0.3583
[3 - Decision Tree] CV F1: 0.3982 | Test F1: 0.3922
[3 - Random Forest] CV F1: 0.4728 | Test F1: 0.4611


## **EXPERIMENT 4: HYPERPARAMETER TUNING**

In [6]:
print("\n--- EXPERIMENT 4: HYPERPARAMETER TUNING ---")
# Random Forest terpilih dari Experiment 3
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf_base, param_grid, cv=cv, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

best_rf = grid_search.best_estimator_
print(f"Hyperparameter Terbaik: {grid_search.best_params_}")

evaluate_model(best_rf, X_train, y_train, X_test, y_test, "4", "Tuned Model (Random Forest)")


--- EXPERIMENT 4: HYPERPARAMETER TUNING ---
Hyperparameter Terbaik: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 100}
[4 - Tuned Model (Random Forest)] CV F1: 0.4801 | Test F1: 0.4616


## **EXPERIMENT 5: ADVANCED MODEL**

In [7]:
print("\n--- EXPERIMENT 5: ADVANCED MODEL ---")
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, eval_metric='logloss')
evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "5", "Advanced Model (XGBoost)")


--- EXPERIMENT 5: ADVANCED MODEL ---
[5 - Advanced Model (XGBoost)] CV F1: 0.4801 | Test F1: 0.4693


## **FINAL SUMMARY TABLE**

In [8]:
print("FINAL SUMMARY OF EXPERIMENT RESULTS")
summary_df = pd.DataFrame(summary_results)
print(summary_df.to_string(index=False))

FINAL SUMMARY OF EXPERIMENT RESULTS
Experiment                          Model  CV F1  Test Accuracy  Test Precision  Test Recall  Test F1
         1 Baseline (Logistic Regression) 0.3367         0.8082          0.6648       0.2675   0.3815
         2       Baseline + Preprocessing 0.3662         0.8083          0.6903       0.2419   0.3583
         3            Logistic Regression 0.3662         0.8083          0.6903       0.2419   0.3583
         3                  Decision Tree 0.3982         0.7195          0.3766       0.4092   0.3922
         3                  Random Forest 0.4728         0.8142          0.6429       0.3595   0.4611
         4    Tuned Model (Random Forest) 0.4801         0.8142          0.6425       0.3602   0.4616
         5       Advanced Model (XGBoost) 0.4801         0.8187          0.6653       0.3625   0.4693
